# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — rebuild the baseline queue and honest features

Reconnects to the warehouse and rebuilds the `w04_baseline_score` ranked queue (rule, reason
codes, `priority_score`) and the `w05_model` honest feature set on the same `month = '2026-03'`
slice, so this notebook can blend them without depending on another notebook's kernel state.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"  # same mid-panel month used across w03/w04/w05/w06 this cycle

print("Connected. Rebuilding the baseline queue and honest feature set below.")

In [ ]:
# --- Rebuild the w04 baseline rule + reason codes + priority_score ---
raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        MODE(ga4_data_available)                                       AS ga4_data_available,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)
df["position_volatility"] = df["position_volatility"].fillna(0.0)

volatility_p90 = df.loc[df["volatility_is_filled"] == 0, "position_volatility"].quantile(0.90)
traffic_p25 = df["total_impressions"].quantile(0.25)

def reason_codes(row):
    codes = []
    if row["active_days"] < 10:
        codes.append("SPARSE_DATA")
    if row["pct_change"] <= -0.20:
        codes.append("DECLINE_20PLUS")
    elif row["pct_change"] >= 0.20:
        codes.append("GROWTH_20PLUS")
    else:
        codes.append("STABLE_TREND")
    if row["volatility_is_filled"] == 0 and row["position_volatility"] >= volatility_p90:
        codes.append("HIGH_VOLATILITY")
    if row["total_impressions"] <= traffic_p25:
        codes.append("LOW_TRAFFIC")
    return codes

df["reason_codes"] = df.apply(reason_codes, axis=1)

def assign_action(codes):
    if "SPARSE_DATA" in codes:
        return "review"
    if "DECLINE_20PLUS" in codes:
        return "declining"
    if "GROWTH_20PLUS" in codes:
        return "growing"
    return "stable"

df["action"] = df["reason_codes"].apply(assign_action)

def confidence_note(row):
    if "SPARSE_DATA" in row["reason_codes"]:
        return f"low — only {row['active_days']} active days observed this month"
    note = f"based on {row['active_days']} active days"
    if "HIGH_VOLATILITY" in row["reason_codes"]:
        note += ", but ranking position is unusually noisy this month"
    if "LOW_TRAFFIC" in row["reason_codes"]:
        note += ", on a small traffic base"
    return note

df["confidence_note"] = df.apply(confidence_note, axis=1)

def priority_score(row):
    if row["action"] != "declining":
        return 0.0
    weight = 1.0
    if "HIGH_VOLATILITY" in row["reason_codes"]:
        weight *= 0.5
    if "LOW_TRAFFIC" in row["reason_codes"]:
        weight *= 0.25
    return round(-row["pct_change"] * np.log1p(row["total_impressions"]) * weight, 4)

df["priority_score"] = df.apply(priority_score, axis=1)
df["reason_codes_str"] = df["reason_codes"].apply(",".join)

ranked = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

print(f"Baseline queue rebuilt: {len(ranked):,} rows.")
print(ranked["action"].value_counts())

In [ ]:
# --- Rebuild the w05 honest-feature logistic regression ---
model_raw = raw.copy()
mdf = model_raw.copy()
mdf["ctr"] = (mdf["total_clicks"] / mdf["total_impressions"]).round(4)
mdf["impressions_per_active_day"] = (mdf["total_impressions"] / mdf["active_days"]).round(2)
mdf["volatility_is_filled"] = mdf["position_volatility"].isna().astype(int)
mdf["position_volatility"] = mdf["position_volatility"].fillna(0.0)
ga4_dummies = pd.get_dummies(mdf["ga4_data_available"], prefix="ga4", dummy_na=True)
mdf = pd.concat([mdf, ga4_dummies], axis=1)
mdf["pct_change"] = (mdf["imp_second_half"] - mdf["imp_first_half"]) / mdf["imp_first_half"]
mdf["is_declining"] = (mdf["pct_change"] < -0.2).astype(int)

HONEST_FEATURES = (["total_impressions", "total_clicks", "avg_position", "active_days",
                     "position_volatility", "volatility_is_filled", "ctr",
                     "impressions_per_active_day"] + list(ga4_dummies.columns))

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, _ = next(gss.split(mdf, groups=mdf["client_hash_id"]))
train_df = mdf.iloc[train_idx]

scaler = StandardScaler().fit(train_df[HONEST_FEATURES])
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(train_df[HONEST_FEATURES]), train_df["is_declining"])

# Score every row — the playbook queue covers the whole month, not just the held-out split.
mdf["model_prob"] = clf.predict_proba(scaler.transform(mdf[HONEST_FEATURES]))[:, 1]

print("Honest-feature model refit on the client-grouped train split. Scored all rows for model_prob.")
print(mdf[["client_hash_id", "content_hash_id", "model_prob"]].head(3))

## 1. Ranked actions + reason codes

**The blend, in plain words:** the rule (`w04_baseline_score`) decides the *action bucket* —
`declining` / `growing` / `stable` / `review` — because it's the transparent, reason-coded logic
a content team can already read. The model (`w05_model`) adds one thing the rule can't: a
predicted probability, from features the rule doesn't fully use together, that this page is
the kind that declines. That probability doesn't override the bucket — it re-ranks *within*
the `declining` bucket, and it adds a new reason code when it disagrees with the rule.

**Final priority for declining rows:**
`final_priority = priority_score * (0.5 + 0.5 * model_prob)` — a row the rule already flagged
gets ranked higher when the model agrees (prob near 1) and discounted, not dropped, when the
model doesn't see the same signal (prob near 0). The rule's own discounts (`HIGH_VOLATILITY`,
`LOW_TRAFFIC`) still apply underneath this — the model multiplies on top, it doesn't replace them.

**New reason codes from the blend:**
- `MODEL_AGREES` — model_prob ≥ 0.5 on a row the rule already called `declining`.
- `MODEL_DISAGREES` — model_prob < 0.3 on a row the rule called `declining`. This is flagged,
  not resolved — the rule and the honest-feature model are reading the same page differently,
  and that disagreement is exactly the kind of row a human should look at before acting, not
  the kind that should get quietly averaged away.
- `MODEL_UNCERTAIN` — model_prob between 0.3 and 0.5 on a `declining` row — the model doesn't
  clearly agree or disagree.

Rows outside the `declining` bucket keep their rule-based action untouched; `model_prob` is
still attached for visibility, but it doesn't change `growing`/`stable`/`review` assignments —
the model was only trained to predict decline, not the other buckets.

In [ ]:
final = ranked.merge(mdf[["client_hash_id", "content_hash_id", "model_prob"]],
                      on=["client_hash_id", "content_hash_id"], how="left")

def blend_row(row):
    if row["action"] != "declining":
        return row["priority_score"], row["reason_codes"]
    codes = list(row["reason_codes"])
    if row["model_prob"] >= 0.5:
        codes.append("MODEL_AGREES")
    elif row["model_prob"] < 0.3:
        codes.append("MODEL_DISAGREES")
    else:
        codes.append("MODEL_UNCERTAIN")
    final_priority = row["priority_score"] * (0.5 + 0.5 * row["model_prob"])
    return final_priority, codes

blended = final.apply(lambda r: pd.Series(blend_row(r), index=["final_priority", "reason_codes"]), axis=1)
final["final_priority"] = blended["final_priority"]
final["reason_codes"] = blended["reason_codes"]
final["reason_codes_str"] = final["reason_codes"].apply(lambda c: ",".join(c))

playbook_queue = final.sort_values("final_priority", ascending=False).reset_index(drop=True)
playbook_queue["rank"] = playbook_queue.index + 1

disagree_share = (playbook_queue["reason_codes_str"].str.contains("MODEL_DISAGREES")).mean()
print(f"Declining rows where rule and model disagree: {disagree_share:.1%}")
playbook_queue[["rank", "content_hash_id", "action", "reason_codes_str",
                "priority_score", "model_prob", "final_priority"]].head(20)

## 2. Intended use and limits

**Who uses this:** a content/SEO team doing a monthly triage pass — deciding which pages to
refresh first, not deciding *what* to change on any given page. Decision-support, not
autopilot: the output is a ranked starting point for a human review, same as the `w04` top-20.

**Where it stops being valid:**
- Scoped to one month (`2026-03`) of `fact_content_daily_performance` only — a page with no
  impressions in this window isn't scored at all, and the queue says nothing about pages
  launched or discontinued outside it.
- The model was trained and validated on this single month's client mix; it has no evidence
  about whether its `model_prob` generalizes to a client vertical, or a future month, it
  hasn't seen — `w06`'s grouped-split check only tells us it isn't just memorizing clients
  *within* this slice.
- No `dim_content` product/content-type signal is in either the rule or the model — a page's
  category, template, or intent isn't part of this ranking at all.
- `is_declining` is defined purely from impressions; a page can lose clicks or conversions
  while impressions hold steady and this queue won't see it, and vice versa.
- `MODEL_AGREES` / `MODEL_DISAGREES` describes agreement between this rule and this model —
  it isn't a statement about which one is *right* on any individual page.

In [ ]:
scope_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(DISTINCT month) AS months
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
print("Confirming the scope this queue actually covers:")
print(scope_check)
print(f"\nRows scored: {len(playbook_queue):,} | Distinct clients: {playbook_queue['client_hash_id'].nunique()}")
print(f"Model-disagreement rows flagged: {(playbook_queue['reason_codes_str'].str.contains('MODEL_DISAGREES')).sum():,}")

## 3. Human review + the no-go list

**Must check before acting on any row:**
- Every `MODEL_DISAGREES` row — the rule and the model read the page differently, and this
  queue doesn't arbitrate that, a person does.
- Every `review` row (`SPARSE_DATA`) before it's dropped from consideration — thin evidence
  cuts both ways, per `w04`'s own caveat.
- Top declining picks against a same-week, same-client dip across *other* pages, per the
  `DECLINE_20PLUS` caveat — a site-wide issue isn't a page-specific refresh decision.

**Never automate from this queue alone:**
- No auto-published content edits or auto-triggered refreshes — every action here is a
  recommendation for a person to evaluate, not an instruction to execute.
- No client-facing message, report, or score generated straight from `reason_codes` without a
  human rewriting it in plain language for that audience.
- No use of `final_priority` or `model_prob` in anything client-billing- or contract-adjacent —
  this queue was built for internal content triage, not for grading client performance.

In [ ]:
needs_human_review = playbook_queue[
    playbook_queue["reason_codes_str"].str.contains("MODEL_DISAGREES|SPARSE_DATA")
]
print(f"Rows requiring human review before any action: {len(needs_human_review):,} "
      f"({len(needs_human_review) / len(playbook_queue):.1%} of the full queue)")
needs_human_review[["rank", "content_hash_id", "action", "reason_codes_str"]].head(10)

## 4. Monitoring / retrain triggers

**Retrain on a fixed cadence, not just on failure:** `is_declining` is defined within a single
month, so a new month of `fact_daily` is a new labeled dataset — retrain when the next month's
panel is available, the same way `w05` was built on `2026-03`.

**Retrain early, before the cadence, if:**
- The grouped-split AUC (per `w06`) drops meaningfully below the measured baseline — a
  concrete number to compare against next cycle, not a vague "if it seems off."
- The `MODEL_DISAGREES` share on a fresh month moves well outside what this month showed —
  rising disagreement between rule and model is itself a signal something upstream shifted.
- The `SPARSE_DATA` share spikes — that points at a data-collection issue, not a scoring one,
  and retraining won't fix it.
- The `declining` / `growing` / `stable` proportions shift sharply month over month — could be
  a real market change or a pipeline break; worth checking which before trusting the new queue.

In [ ]:
action_shares = playbook_queue["action"].value_counts(normalize=True).round(3)
disagree_share = (playbook_queue["reason_codes_str"].str.contains("MODEL_DISAGREES")).mean()
sparse_share = (playbook_queue["reason_codes_str"].str.contains("SPARSE_DATA")).mean()

print("This month's baseline numbers to compare future runs against:")
print(action_shares)
print(f"\nMODEL_DISAGREES share: {disagree_share:.1%}")
print(f"SPARSE_DATA share: {sparse_share:.1%}")
print("\nRecord these three numbers somewhere durable (this printout, or work/outputs/) —")
print("next month's run should be compared against them, not against a hardcoded guess.")

## 5. Exports for the paper

Writes the final ranked queue to `work/outputs/` — your paper builds on these files.

In [ ]:
import os
os.makedirs("work/outputs", exist_ok=True)

export_cols = ["rank", "client_hash_id", "content_hash_id", "action", "reason_codes_str",
               "priority_score", "model_prob", "final_priority", "confidence_note"]
playbook_queue[export_cols].to_csv("work/outputs/w07_playbook_queue.csv", index=False)

print(f"Wrote {len(playbook_queue):,} rows to work/outputs/w07_playbook_queue.csv")
playbook_queue[export_cols].head(3)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.